In [24]:
import pyodbc
import pandas as pd

# 1. Cargar Excel y construir la llave concatenada id = Id_Num_Conv + Id_Num_Ver
excel_path = "input/dbo_C_ccConvEscImpPorc_Soriana_20200720.xlsx"
df_excel = pd.read_excel(excel_path)
df_excel["id"] = df_excel["Id_Num_Conv"].astype(str) + df_excel["Id_Num_Ver"].astype(str)

print("Excel cargado:", df_excel.shape)
print("Columna id generada (primeros 5):", df_excel["id"].head().tolist())

Excel cargado: (5316, 9)
Columna id generada (primeros 5): ['10461', '10461', '10461', '10470', '10470']


In [25]:
# 2. Valores unicos de la llave concatenada
values = df_excel["id"].dropna().unique().tolist()
print("Valores unicos del Excel:", len(values))

# 3. Conexion SQL Server
conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=ATL20AF2222SQ19;"
    "DATABASE=SORIANA_MX_2024_PROD_F;"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(conn_str)

# 4. Consulta en batches usando la llave concatenada en SQL (ID_NUM_CONV + ID_NUM_VER)
batch_size = 500
resultados = []

tabla_sql = "dbo.Convenios_Legado_Soriana"
cols_sql = [
    "ID_NUM_CONV",
    "ID_NUM_VER",
    "ID_NUM_PLAZOPAGO",
    "DESC_CONVEVTO",
    "FECINI",
    "FECFIN",
    "ID_NUM_PROV",
    "DESC_CONVSTAT_NVO",
    "DESC_CONVEVTO_NVO"
]
cols_select = ", ".join(cols_sql)

for i in range(0, len(values), batch_size):
    subset = values[i : i + batch_size]
    placeholders = ",".join("?" for _ in subset)

    query = f"""
        SELECT
            CONCAT(CAST(ID_NUM_CONV AS varchar(50)), CAST(ID_NUM_VER AS varchar(50))) AS id,
            {cols_select}
        FROM {tabla_sql}
        WHERE CONCAT(CAST(ID_NUM_CONV AS varchar(50)), CAST(ID_NUM_VER AS varchar(50))) IN ({placeholders})
    """

    print("Batch", i//batch_size + 1)
    df_tmp = pd.read_sql(query, conn, params=subset)
    resultados.append(df_tmp)

# 5. Union final contra el Excel
df_sql = pd.concat(resultados, ignore_index=True) if resultados else pd.DataFrame()
print("Filas SQL recuperadas:", len(df_sql))

df_final = df_excel.merge(df_sql, on="id", how="left", suffixes=("_excel", "_sql"))
print("Filas finales (merge):", len(df_final))

df_final.head()

Valores unicos del Excel: 1316
Batch 1


C:\Users\opined01\AppData\Local\Temp\2\ipykernel_15092\1955019215.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tmp = pd.read_sql(query, conn, params=subset)


Batch 2
Batch 3
Filas SQL recuperadas: 634
Filas finales (merge): 5332


,Id,Id_Num_Conv,Id_Num_Ver,Id_Cnsc_Esc,Imp_LimInf,Imp_LimSup,Num_Subtipo,Porc_Descontar,id,ID_NUM_CONV,ID_NUM_VER,ID_NUM_PLAZOPAGO,DESC_CONVEVTO,FECINI,FECFIN,ID_NUM_PROV,DESC_CONVSTAT_NVO,DESC_CONVEVTO_NVO
0,10461,1046,1,1,0.00,400000.00,1,0.0,10461,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10461,1046,1,2,400000.01,449999.00,1,0.5,10461,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10461,1046,1,3,449999.01,450000.00,1,1.0,10461,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10470,1047,0,1,0.00,19999.99,1,0.0,10470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10470,1047,0,2,20000.00,25000.00,1,0.5,10470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
